# 02 - Figure 1 and Supplementary Table S1


In [ ]:
# --- config ---
HIDALGO_RCA_INPUT = "counts"  # "counts" | "binary"
HIDALGO_GRAPH = "plot_graph"  # "plot_graph" | "full_phi" | "disparity_backbone"

NULL_CATEGORY = "Prescription"
NULL_PERMUTATIONS = 5000
NULL_RANDOM_SEED = 42

import os
import sys
import importlib
from pathlib import Path
from typing import Dict, Tuple

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").is_dir() else Path.cwd()
sys.path.insert(0, str(NOTEBOOK_DIR))
import dnm_viz_utils as dvu
dvu = importlib.reload(dvu)

repo_root = dvu.repo_root
load_master_drugs = dvu.load_master_drugs
load_category2_most_common_mapping = dvu.load_category2_most_common_mapping
load_category_color_map = dvu.load_category_color_map
MARKETS = dvu.MARKETS
MARKET_FILE_NAMES = dvu.MARKET_FILE_NAMES
style_axes = dvu.style_axes

os.chdir(repo_root())
ROOT = repo_root()
sys.path.insert(0, str(ROOT / "scripts"))
from core_periphery_utils import compute_borgatti_everett_core_score
OUT = ROOT / "prescription_reports/new_category2_networks"
TAB = OUT / "tables"
NET = OUT / "networks"
FIG = OUT / "figures"
TAB.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

def hidalgo_suffix() -> str:
    return "" if HIDALGO_RCA_INPUT == "counts" else f"_{HIDALGO_RCA_INPUT}"

df_all = load_master_drugs()
df_all = df_all[df_all["Website"].isin(MARKETS)]
prod_to_cat2 = load_category2_most_common_mapping(df_all)
color_map = load_category_color_map(prod_to_cat2)
print(f"Hidalgo product space: {HIDALGO_RCA_INPUT}/{HIDALGO_GRAPH}")


In [ ]:
def giant_component(G: nx.Graph) -> nx.Graph:
    if G.number_of_nodes() == 0:
        return G.copy()
    if nx.is_connected(G):
        return G.copy()
    return G.subgraph(max(nx.connected_components(G), key=len)).copy()






def load_saved_graphml(path: Path) -> nx.Graph | None:
    if not path.exists():
        return None
    G = nx.read_graphml(path)
    return giant_component(nx.relabel_nodes(G, {n: str(n) for n in G.nodes()}))




def add_listing_id(df: pd.DataFrame) -> pd.Series:
    website = df["Website"].fillna("").astype(str).str.strip()
    pages = df["Pages"].fillna("").astype(str).str.strip()
    vendor = df["ID_Num"].fillna("").astype(str).str.strip()
    names = df["Productname"].fillna("").astype(str).str.strip().str.casefold()
    cat3 = df["Category 3"].fillna("").astype(str).str.strip()
    fallback = website + " | " + vendor + " | " + names + " | " + cat3
    valid_pages = pages.ne("") & ~pages.str.lower().isin(["nan", "none"])
    out = pages.where(valid_pages, fallback)
    return out.where(~website.isin(["WhiteHouse Market", "ToRReZ"]), fallback).astype(str).str.strip()


def compute_shares_for_market(market: str, df_drugs: pd.DataFrame, prod_to_cat2: dict) -> pd.DataFrame | None:
    G = get_network_for_market(market, df_drugs)
    if G is None or G.number_of_nodes() == 0:
        return None
    G = nx.relabel_nodes(G, {n: str(n) for n in G.nodes()})
    node_cat = {n: prod_to_cat2.get(str(n), "Unknown") for n in G.nodes()}
    core_scores, _ = compute_borgatti_everett_core_score(G)
    if core_scores is None:
        return None
    market_data = df_drugs[df_drugs["Website"] == market].copy()
    market_data["cat2_most_common"] = market_data["Category 3"].astype(str).str.strip().map(lambda p: prod_to_cat2.get(p, "Unknown"))
    market_data["_listing_id"] = add_listing_id(market_data)
    listings = market_data.drop_duplicates(["Website", "_listing_id"])
    cat_listing = listings.groupby("cat2_most_common")["_listing_id"].nunique().to_dict()
    total_listings = listings["_listing_id"].nunique()
    product_shares = {c: cat_listing.get(c, 0) / total_listings if total_listings else 0.0 for c in set(prod_to_cat2.values())}
    vendor_categories = {}
    for vid, vg in market_data.groupby("ID_Num"):
        vprods = set(vg["Category 3"].dropna().astype(str).str.strip())
        vcats = {prod_to_cat2.get(p, "Unknown") for p in vprods} - {"Unknown"}
        for c in vcats:
            vendor_categories.setdefault(c, set()).add(vid)
    total_v = market_data["ID_Num"].nunique()
    vendor_shares = {c: len(s) / total_v if total_v else 0.0 for c, s in vendor_categories.items()}
    cat_core = {}
    for node, sc in core_scores.items():
        c = node_cat.get(node, "Unknown")
        cat_core.setdefault(c, []).append(sc)
    tot_core = sum(core_scores.values())
    core_share = {c: sum(v) / tot_core if tot_core > 0 else 0.0 for c, v in cat_core.items()}
    all_cats = set(product_shares) | set(vendor_shares) | set(core_share)
    rows = []
    for cat in all_cats:
        rows.append({"market": market, "category": cat, "product_share": product_shares.get(cat, 0.0), "vendor_share": vendor_shares.get(cat, 0.0), "core_score_share": core_share.get(cat, 0.0)})
    return pd.DataFrame(rows)


def graph_core_category_frame(market: str, df_drugs: pd.DataFrame, prod_to_cat2: dict) -> pd.DataFrame | None:
    G = get_network_for_market(market, df_drugs)
    if G is None or G.number_of_nodes() == 0:
        return None
    G = nx.relabel_nodes(G, {n: str(n) for n in G.nodes()})
    core_scores, _ = compute_borgatti_everett_core_score(G)
    if core_scores is None:
        return None
    return pd.DataFrame({
        "market": market,
        "product_cat3": list(core_scores.keys()),
        "category": [prod_to_cat2.get(str(n), "Unknown") for n in core_scores.keys()],
        "core_score": list(core_scores.values()),
    })


def core_share_label_permutation_test(
    core_df: pd.DataFrame,
    target_category: str = NULL_CATEGORY,
    n_permutations: int = NULL_PERMUTATIONS,
    seed: int = NULL_RANDOM_SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    summary_rows = []
    dist_rows = []
    for market, mdf in core_df.groupby("market", sort=False):
        scores = mdf["core_score"].to_numpy(dtype=float)
        labels = mdf["category"].astype(str).to_numpy()
        total_core = scores.sum()
        n_nodes = len(scores)
        n_target = int((labels == target_category).sum())
        observed = float(scores[labels == target_category].sum() / total_core) if total_core > 0 else np.nan
        null_values = np.empty(n_permutations, dtype=float)
        for i in range(n_permutations):
            shuffled = rng.permutation(labels)
            null_values[i] = float(scores[shuffled == target_category].sum() / total_core) if total_core > 0 else np.nan
        null_mean = float(np.nanmean(null_values))
        null_sd = float(np.nanstd(null_values, ddof=1))
        z = float((observed - null_mean) / null_sd) if null_sd > 0 else np.nan
        p_upper = float((np.sum(null_values >= observed) + 1) / (n_permutations + 1))
        q025, q975 = np.nanquantile(null_values, [0.025, 0.975])
        summary_rows.append({
            "network_kind": "hidalgo",
            "network_label": f"hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}",
            "market": market,
            "target_category": target_category,
            "n_nodes": n_nodes,
            "n_target_nodes": n_target,
            "observed_core_share": observed,
            "null_mean_core_share": null_mean,
            "null_sd_core_share": null_sd,
            "null_q025_core_share": float(q025),
            "null_q975_core_share": float(q975),
            "z_score": z,
            "empirical_p_upper": p_upper,
            "n_permutations": n_permutations,
            "seed": seed,
        })
        dist_rows.extend({
            "network_kind": "hidalgo",
            "network_label": f"hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}",
            "market": market,
            "target_category": target_category,
            "permutation": i,
            "null_core_share": float(v),
        } for i, v in enumerate(null_values, start=1))
    return pd.DataFrame(summary_rows), pd.DataFrame(dist_rows)


def get_network_for_market(market: str, df_drugs: pd.DataFrame) -> nx.Graph:
    tag = MARKET_FILE_NAMES[market]
    suffix = hidalgo_suffix()
    path = NET / f"{tag}_hidalgo{suffix}_{HIDALGO_GRAPH}.graphml"
    graph = load_saved_graphml(path)
    if graph is None:
        raise FileNotFoundError(f"Missing {path}. Run notebook 01 first.")
    return graph


In [ ]:
def add_subplot_label(ax, idx: int):
    ax.text(-0.08, 1.02, chr(65 + idx), transform=ax.transAxes, ha="left", va="bottom", fontsize=18)


def panel_a_total_listings(df_drugs: pd.DataFrame, prod_to_cat2: dict, color_map: dict):
    exclude = {
        "Custom Listing", "Other", "Paraphernalia", "Tobacco", "Precursors",
        "Testing Kits", "Laboratory Equipment", "Bags and Packages", "Bags And Packages",
    }
    category_listings = {}
    for market in MARKETS:
        md = df_drugs[df_drugs["Website"] == market].copy()
        md["cat2_most_common"] = md["Category 3"].astype(str).str.strip().map(lambda p: prod_to_cat2.get(p, "Unknown"))
        md["_listing_id"] = add_listing_id(md)
        listings = md.drop_duplicates(["Website", "_listing_id"])
        cat2c = listings.groupby("cat2_most_common")["_listing_id"].nunique()
        for cat, cnt in cat2c.items():
            if cat in exclude:
                continue
            category_listings[cat] = category_listings.get(cat, 0) + int(cnt)
    df_bar = pd.DataFrame([{"category": k, "total_listings": v} for k, v in category_listings.items()])
    df_bar = df_bar.sort_values("total_listings", ascending=True)
    df_bar["category_display"] = df_bar["category"].replace("Benzos", "Benzodiazepines")
    ax = plt.gca()
    colors = [color_map.get(c, "#B3B3B3") for c in df_bar["category"]]
    ax.barh(df_bar["category_display"], df_bar["total_listings"], color=colors, edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Reviewed listing proxies", fontsize=13)
    style_axes(ax)
    add_subplot_label(ax, 0)


def panel_b_scatter(df_results: pd.DataFrame, color_map: dict, y_label: str):
    platform_shapes = {"WhiteHouse Market": "o", "Dark0de": "s", "ToRReZ": "D", "Versus": "^"}
    ax = plt.gca()
    x_max, y_max = 0.6, 1.0
    xms, yms = np.sqrt(x_max), np.sqrt(y_max)
    for _, row in df_results.iterrows():
        cat, mkt = row["category"], row["market"]
        x = np.sqrt(row["product_share"])
        y = np.sqrt(row["core_score_share"])
        ax.scatter(
            x, y, c=[color_map.get(cat, "#B3B3B3")], marker=platform_shapes.get(mkt, "o"),
            s=60, alpha=0.7, edgecolors="black", linewidth=0.5,
        )
        if cat == "Prescription":
            if mkt in ("WhiteHouse Market", "Versus"):
                ax.annotate(cat, (x, y), xytext=(5, 0), textcoords="offset points", fontsize=10, alpha=0.8, ha="left", va="center")
            else:
                ax.annotate(cat, (x, y), xytext=(-5, 0), textcoords="offset points", fontsize=10, alpha=0.8, ha="right", va="center")
    diag = min(xms, yms)
    ax.plot([0, diag], [0, diag], "k--", alpha=0.3, lw=1)
    ax.set_xlim(0, xms)
    ax.set_ylim(0, yms)
    xt = np.linspace(0, xms, 6)
    ax.set_xticks(xt)
    ax.set_xticklabels([f"{v**2:.2f}" for v in xt])
    yt = np.linspace(0, yms, 6)
    ax.set_yticks(yt)
    ax.set_yticklabels([f"{v**2:.2f}" for v in yt])
    ax.set_xlabel("Reviewed listing share", fontsize=13)
    ax.set_ylabel(y_label, fontsize=13)
    from matplotlib.lines import Line2D
    leg_labs = {"WhiteHouse Market": "WhiteHouse\nMarket", "Dark0de": "Dark0de", "ToRReZ": "ToRReZ", "Versus": "Versus"}
    h = [Line2D([0], [0], marker=platform_shapes[m], color="w", markerfacecolor="gray", markersize=10, label=leg_labs.get(m, m), markeredgecolor="black", markeredgewidth=0.5) for m in MARKETS]
    ax.legend(handles=h, loc="upper right", frameon=False, fontsize=11)
    style_axes(ax)
    add_subplot_label(ax, 1)


parts = [compute_shares_for_market(m, df_all, prod_to_cat2) for m in MARKETS]
parts = [p for p in parts if p is not None]
if not parts:
    raise RuntimeError("No results — check backbone CSVs or run notebook 02 for Hidalgo.")
df_res = pd.concat(parts, ignore_index=True)
ylab = "Core score share"
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plt.sca(ax1)
panel_a_total_listings(df_all, prod_to_cat2, color_map)
plt.sca(ax2)
panel_b_scatter(df_res, color_map, ylab)
plt.tight_layout()
fn = f"figure1_hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}_core_share_nb.png"
out = FIG / fn
plt.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
plt.close()
df_res.to_csv(TAB / fn.replace(".png", "_results.csv"), index=False)
print("Saved", out)

## Core-share label-permutation null model


In [ ]:
core_parts = [graph_core_category_frame(m, df_all, prod_to_cat2) for m in MARKETS]
core_parts = [p for p in core_parts if p is not None]
if not core_parts:
    raise RuntimeError("No core-score frames available for the null model.")
core_category_df = pd.concat(core_parts, ignore_index=True)
null_summary, null_distribution = core_share_label_permutation_test(core_category_df)

null_prefix = f"figure1_hidalgo{hidalgo_suffix()}_{HIDALGO_GRAPH}_core_share_label_permutation_null"

summary_path = TAB / f"{null_prefix}_summary.csv"
dist_path = TAB / f"{null_prefix}_distribution.csv"
null_summary.to_csv(summary_path, index=False)
null_distribution.to_csv(dist_path, index=False)
print("Saved null-model summary:", summary_path)
print("Saved null-model distribution:", dist_path)
display(null_summary[[
    "market",
    "target_category",
    "n_nodes",
    "n_target_nodes",
    "observed_core_share",
    "null_mean_core_share",
    "null_q025_core_share",
    "null_q975_core_share",
    "z_score",
    "empirical_p_upper",
]])


In [ ]:
df_res[df_res['category']=='Prescription']